# 🛡️ CodeSentinel AI — Train Model 2: Semantic Clone & Redundancy Detector
### Fine-Tuning CodeBERT into a Siamese SentenceTransformer on BigCloneBench (Free T4 GPU in Google Colab)

This notebook trains **Model 2 (The Semantic Clone & Redundancy Detector)** on the **CodeXGLUE BigCloneBench** dataset.

* **Task**: Siamese Bi-Encoder (Encodes code snippets into 768-d embeddings; computes Cosine Similarity to detect copy-pasted or redundant logic)
* **Hardware**: Free Google Colab T4 GPU (16 GB VRAM)
* **Base Model**: `microsoft/codebert-base` (pre-trained on 6 programming languages)
* **Dataset**: `google/code_x_glue_cc_clone_detection_big_clone_bench` (20,000 balanced pairs)
* **Target Hub Repo**: `wraithklu/codesentinel-clone-detector`

## 1. Verify GPU Hardware
Ensure you have selected **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
!pip install --quiet sentence-transformers datasets accelerate huggingface_hub

## 3. Load BigCloneBench & Prepare Balanced Pairs

In [ ]:
import random
from datasets import load_dataset
from sentence_transformers import InputExample

print("📥 Loading CodeXGLUE BigCloneBench dataset...")
dataset = load_dataset("google/code_x_glue_cc_clone_detection_big_clone_bench")

# Sample 10,000 balanced pairs (~3-5 mins training) (10k clones, 10k non-clones) for optimal fast training
TRAIN_SAMPLES = 10000
TARGET_EACH = TRAIN_SAMPLES // 2

train_clones = []
train_non_clones = []

print("⚖️ Balancing training pairs...")
for row in dataset["train"]:
    if len(train_clones) >= TARGET_EACH and len(train_non_clones) >= TARGET_EACH:
        break
    f1 = row.get("func1", "").strip()
    f2 = row.get("func2", "").strip()
    is_clone = bool(row.get("label", False))

    if len(f1) > 20 and len(f2) > 20:
        if is_clone and len(train_clones) < TARGET_EACH:
            train_clones.append(InputExample(texts=[f1, f2], label=1.0))
        elif not is_clone and len(train_non_clones) < TARGET_EACH:
            train_non_clones.append(InputExample(texts=[f1, f2], label=0.0))

train_examples = train_clones + train_non_clones
random.seed(42)
random.shuffle(train_examples)
print(f"✅ Training set ready: {len(train_examples)} pairs (50% Clones, 50% Non-Clones)")

# Validation set (2,000 balanced pairs)
VAL_SAMPLES = 2000
VAL_TARGET = VAL_SAMPLES // 2
val_clones, val_non_clones = [], []

for row in dataset["validation"]:
    if len(val_clones) >= VAL_TARGET and len(val_non_clones) >= VAL_TARGET:
        break
    f1 = row.get("func1", "").strip()
    f2 = row.get("func2", "").strip()
    is_clone = bool(row.get("label", False))

    if len(f1) > 20 and len(f2) > 20:
        if is_clone and len(val_clones) < VAL_TARGET:
            val_clones.append((f1, f2, 1.0))
        elif not is_clone and len(val_non_clones) < VAL_TARGET:
            val_non_clones.append((f1, f2, 0.0))

val_data = val_clones + val_non_clones
val_s1 = [x[0] for x in val_data]
val_s2 = [x[1] for x in val_data]
val_scores = [x[2] for x in val_data]
print(f"✅ Validation set ready: {len(val_data)} pairs")

## 4. Construct Siamese CodeBERT Architecture

In [ ]:
from sentence_transformers import SentenceTransformer, models

BASE_MODEL = "microsoft/codebert-base"
print(f"🏗️ Building Siamese architecture on top of '{BASE_MODEL}'...")

# 1. Transformer token encoder
word_embedding_model = models.Transformer(BASE_MODEL, max_seq_length=256)

# 2. Mean Pooling layer (maps token hidden states into a single 768-d vector)
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False,
)

# 3. Combine into unified SentenceTransformer
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
print(model)

## 5. Configure DataLoader & Cosine Similarity Loss

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import losses, evaluation

BATCH_SIZE = 32
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

# CosineSimilarityLoss enforces cosine(f1, f2) -> 1.0 for clones, -> 0.0 for non-clones
train_loss = losses.CosineSimilarityLoss(model=model)

# Evaluator computes Spearman & Pearson rank correlation on validation set
evaluator = evaluation.EmbeddingSimilarityEvaluator(
    val_s1, val_s2, val_scores,
    name="bigclonebench-validation",
    show_progress_bar=False
)
print("✅ Training objectives & evaluator ready!")

## 6. Train the Siamese Clone Detector (Estimated: ~18-22 mins on T4 GPU)

In [ ]:
EPOCHS = 2
warmup_steps = int(len(train_dataloader) * EPOCHS * 0.1)
print(f"🚀 Starting training for {EPOCHS} epochs ({warmup_steps} warmup steps)...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    evaluation_steps=len(train_dataloader) // 2,
    warmup_steps=warmup_steps,
    output_path="./codesentinel-clone-model",
    show_progress_bar=True,
    use_amp=True,  # ⚡ 16-bit Mixed Precision on T4 GPU (3x-4x speedup)
)
print("🎉 Training finished successfully!")

## 7. Test Clone Detection on Live Code Snippets

In [ ]:
from sentence_transformers import util

# Test Pair 1: Semantic Clones (Bubble Sort vs Insertion Sort)
code_a = """
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr
"""

code_b = """
def sort_array_ascending(numbers):
    size = len(numbers)
    for x in range(size):
        for y in range(0, size - x - 1):
            if numbers[y] > numbers[y + 1]:
                temp = numbers[y]
                numbers[y] = numbers[y + 1]
                numbers[y + 1] = temp
    return numbers
"""

# Test Pair 2: Completely Unrelated (Database query)
code_c = """
def get_user_by_id(db_session, user_id):
    return db_session.query(User).filter(User.id == user_id).first()
"""

embeddings = model.encode([code_a, code_b, code_c], convert_to_tensor=True)

sim_ab = util.cos_sim(embeddings[0], embeddings[1]).item() * 100
sim_ac = util.cos_sim(embeddings[0], embeddings[2]).item() * 100

print(f"📊 Similarity (BubbleSort vs Renamed BubbleSort): {sim_ab:.2f}%")
print(f"📊 Similarity (BubbleSort vs DB Query):           {sim_ac:.2f}%")

if sim_ab >= 85.0:
    print("\n✅ SUCCESS: Model correctly identified Code A and Code B as SEMANTIC CLONES!")

## 8. Export & Push to Hugging Face Model Hub

In [ ]:
from huggingface_hub import notebook_login

print("🔑 Sign in to Hugging Face:")
notebook_login()

In [ ]:
HF_USERNAME = "wraithklu"  # Your Hugging Face username
REPO_NAME = f"{HF_USERNAME}/codesentinel-clone-detector"

print(f"📤 Pushing clone detector model to: https://huggingface.co/{REPO_NAME}")
model.save("./codesentinel-clone-model")
model.push_to_hub(REPO_NAME)

print(f"🎉 SUCCESS! Model 2 is live at https://huggingface.co/{REPO_NAME}")